# 🎬 Clipped AI Studio — Google Colab One-Click Cloud Deployment

Welcome to **Clipped AI Studio**, the high-performance AI video generation and faceless short-form video creation platform built with **Next.js 16, React 19, FFmpeg, and multi-provider AI engines**.

---

### ✨ Included Studio Capabilities:
- 🎙️ **Multi-Provider TTS Engine**: ElevenLabs Multilingual v2, Google Cloud TTS (Neural2/Journey), Coqui TTS, and offline deterministic RIFF/WAVE PCM synthesizer. Supports English + 6 Indian languages (Hindi, Tamil, Telugu, Kannada, Bengali, Marathi).
- 🎥 **AI Video Generation**: Kling AI v1, Luma Dream Machine, and Fal.ai Flux with prompt optimization.
- 🎵 **FFmpeg Audio Mixing**: Sidechain compression speech ducking, background music loops, volume normalization, and video muxing.
- 📱 **Multi-Platform Publishing**: YouTube Data API v3, Instagram Graph API (Reels), and TikTok Content API with rate limiting and exponential backoff.
- ⚡ **Zero-Cost Dry-Run Mode**: Full functionality can be tested offline without incurring third-party API costs.

---

### 🚀 Recommended Hardware:
- **GPU (T4 / V100 / A100)**: Recommended for local models and fast video rendering (`Runtime` > `Change runtime type` > `T4 GPU`).
- **CPU**: Fully supported for all API-driven modes, audio mixing, and dry-run testing.

---

### ⏱️ Setup Instructions:
Run **Cells 2 through 7** sequentially. Cell 7 will provide your **Public Tunnel URL** and **Tunnel Password**.

In [ ]:
# ==============================================================================
# Cell 2: Hardware & Environment Diagnostics
# ==============================================================================
import os
import sys
import platform
import subprocess

print("=" * 60)
print("🔍 CLIPPED AI STUDIO — ENVIRONMENT DIAGNOSTICS")
print("=" * 60)

# OS & Python Information
print(f"🖥️  OS: {platform.system()} {platform.release()} ({platform.version()})")
print(f"🐍 Python: {sys.version.split()[0]} ({sys.executable})")
print(f"⚙️  CPU Cores: {os.cpu_count()}")

# Memory Information
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / (1024 ** 3)
    print(f"🧠 System RAM: {ram_gb:.2f} GB")
except ImportError:
    pass

# GPU Information
print("-" * 60)
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"🚀 GPU Detected: {gpu_name} ({gpu_vram:.2f} GB VRAM)")
        print(f"🔥 CUDA Version: {torch.version.cuda}")
    else:
        print("ℹ️  GPU: Not available (Running in CPU Mode — API & Mock modes fully functional)")
except ImportError:
    print("ℹ️  PyTorch not imported; checking nvidia-smi...")
    res = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if res.returncode == 0:
        print(res.stdout.split("\n")[2])
    else:
        print("ℹ️  No GPU attached (CPU mode).")

print("=" * 60)
print("✅ Diagnostics completed successfully.")

In [ ]:
%%bash
set -e

# ==============================================================================
# Cell 3: System Dependencies Installation (Node.js 20, pnpm, FFmpeg, localtunnel)
# ==============================================================================
echo "📦 Step 1/4: Installing System Dependencies (FFmpeg, Curl, Git)..."
apt-get update -qq > /dev/null
apt-get install -y -qq ffmpeg curl git lsof > /dev/null

echo "🟢 Step 2/4: Installing Node.js 20.x LTS via NodeSource..."
curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
apt-get install -y -qq nodejs > /dev/null

echo "⚡ Step 3/4: Installing pnpm and localtunnel globally..."
npm install -g pnpm@11.24.0 localtunnel > /dev/null 2>&1

echo "------------------------------------------------------------"
echo "✅ Installed System Tooling Versions:"
echo "• Node.js:     $(node -v)"
echo "• npm:         v$(npm -v)"
echo "• pnpm:        v$(pnpm -v)"
echo "• FFmpeg:      $(ffmpeg -version | head -n 1)"
echo "• Localtunnel: $(lt --version 2>/dev/null || echo 'Installed')"
echo "------------------------------------------------------------"

In [ ]:
# ==============================================================================
# Cell 4: Workspace & Project Directory Setup
# ==============================================================================
import os
import sys

# Target directory in Google Colab environment
PROJECT_DIR = "/content/clipped"

if not os.path.exists(PROJECT_DIR):
    if os.path.exists("/content") and os.path.exists("./package.json"):
        # Current directory is already the project root
        PROJECT_DIR = os.path.abspath(".")
    else:
        print(f"📁 Setting up project directory at {PROJECT_DIR}...")
        os.makedirs(PROJECT_DIR, exist_ok=True)
        # Note: If running standalone in Colab, clone repo or link workspace
        # subprocess.run(["git", "clone", "https://github.com/your-repo/clipped.git", PROJECT_DIR])

os.chdir(PROJECT_DIR)
print(f"📍 Working Directory: {os.getcwd()}")

# Verify package.json presence
if os.path.exists(os.path.join(PROJECT_DIR, "package.json")):
    print("✅ Found package.json in workspace.")
else:
    print("⚠️  Warning: package.json not found in current directory. Please ensure project files are loaded.")

In [ ]:
# ==============================================================================
# Cell 5: Environment Variables Configuration (.env.local)
# ==============================================================================
# @title ⚙️ Configure Studio Environment Settings { display-mode: "form" }
import os
import secrets
import subprocess

# @markdown ### 🛡️ Cost-Safety & Execution Mode
ENABLE_DRY_RUN_MODE = True  # @param {type:"boolean"}
# @markdown *(Keep True to test TTS, AI Video, and Social Publishing with zero API costs)*

# @markdown ---
# @markdown ### 🔑 AI Provider API Keys (Optional - Leave blank for Cost-Safe Mock Fallbacks)
ELEVENLABS_API_KEY = ""  # @param {type:"string"}
GOOGLE_TTS_API_KEY = ""  # @param {type:"string"}
KLING_API_KEY = ""  # @param {type:"string"}
LUMA_API_KEY = ""  # @param {type:"string"}
FAL_API_KEY = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 🗄️ Supabase Configuration (Optional - Defaults to local/mock store)
NEXT_PUBLIC_SUPABASE_URL = "https://mock.supabase.co"  # @param {type:"string"}
NEXT_PUBLIC_SUPABASE_ANON_KEY = "mock-anon-key-clipped-studio-2026"  # @param {type:"string"}

# Generate a cryptographically secure NextAuth secret
NEXTAUTH_SECRET = secrets.token_hex(32)
NEXTAUTH_URL = "http://localhost:3000"

date_str = "Colab Session"
try:
    date_str = subprocess.run(["date", "-u"], capture_output=True, text=True).stdout.strip()
except Exception:
    pass

env_content = f"""# Clipped AI Studio — Google Colab Generated Configuration
# Generated on: {date_str}

# Core NextAuth & App Settings
NEXTAUTH_SECRET="{NEXTAUTH_SECRET}"
NEXTAUTH_URL="{NEXTAUTH_URL}"
NODE_ENV="development"
DRY_RUN_MODE="{str(ENABLE_DRY_RUN_MODE).lower()}"

# Supabase Storage & Database
NEXT_PUBLIC_SUPABASE_URL="{NEXT_PUBLIC_SUPABASE_URL}"
NEXT_PUBLIC_SUPABASE_ANON_KEY="{NEXT_PUBLIC_SUPABASE_ANON_KEY}"

# TTS Providers
ELEVENLABS_API_KEY="{ELEVENLABS_API_KEY}"
GOOGLE_TTS_API_KEY="{GOOGLE_TTS_API_KEY}"

# AI Video Generation Models
KLING_API_KEY="{KLING_API_KEY}"
LUMA_API_KEY="{LUMA_API_KEY}"
FAL_API_KEY="{FAL_API_KEY}"
"""

with open(".env.local", "w") as f:
    f.write(env_content)

print("=" * 60)
print("✅ .env.local successfully written!")
print(f"• Dry-Run Mode: {'ENABLED (Cost-Safe Mocking)' if ENABLE_DRY_RUN_MODE else 'LIVE API CALLS'}")
print(f"• NextAuth Secret: Generated (32 bytes)")
print("• AI Keys Configured:", [k for k, v in [
    ("ElevenLabs", ELEVENLABS_API_KEY),
    ("Google TTS", GOOGLE_TTS_API_KEY),
    ("Kling", KLING_API_KEY),
    ("Luma", LUMA_API_KEY),
    ("Fal.ai", FAL_API_KEY)
] if v])
print("=" * 60)

In [ ]:
%%bash
set -e

# ==============================================================================
# Cell 6: Project Dependencies Installation
# ==============================================================================
echo "📦 Installing project dependencies via pnpm..."
pnpm install --prefer-offline 2>&1 | tail -n 15

echo "------------------------------------------------------------"
echo "✅ Dependencies installed successfully."

In [ ]:
# ==============================================================================
# Cell 7: Start Next.js Background Server & Launch Public Tunnel
# ==============================================================================
import time
import urllib.request
import subprocess
import os

print("=" * 60)
print("🚀 LAUNCHING CLIPPED AI STUDIO & PUBLIC TUNNEL")
print("=" * 60)

# Step 1: Clean up any stale processes on port 3000
subprocess.run("fuser -k 3000/tcp > /dev/null 2>&1 || true", shell=True)
time.sleep(1)

# Step 2: Start Next.js background server
log_file = open("server.log", "w")
print("🌐 [1/3] Starting Next.js development server (port 3000)...\n")
server_process = subprocess.Popen(
    ["pnpm", "run", "dev"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=dict(os.environ, PORT="3000")
)

# Step 3: Wait for Healthcheck API (http://localhost:3000/api/health)
print("⏳ [2/3] Waiting for server readiness...")
server_ready = False
for attempt in range(35):
    try:
        with urllib.request.urlopen("http://localhost:3000/api/health", timeout=2) as resp:
            if resp.status == 200:
                server_ready = True
                print("   ✨ Next.js server is ONLINE and responding to /api/health!")
                break
    except Exception:
        time.sleep(1)

if not server_ready:
    print("❌ Server failed to respond within 35 seconds. Displaying last 20 log lines:")
    log_file.flush()
    with open("server.log", "r") as f:
        print("".join(f.readlines()[-20:]))
    raise RuntimeError("Next.js server startup timed out.")

# Step 4: Retrieve public IP endpoint password for localtunnel
print("🔑 [3/3] Fetching public tunnel endpoint password...")
tunnel_password = "Unavailable"
try:
    with urllib.request.urlopen("https://loca.lt/mytunnelpassword", timeout=5) as resp:
        tunnel_password = resp.read().decode("utf-8").strip()
except Exception:
    try:
        with urllib.request.urlopen("https://ipv4.icanhazip.com", timeout=5) as resp:
            tunnel_password = resp.read().decode("utf-8").strip()
    except Exception:
        pass

# Step 5: Start localtunnel process
print("\n" + "=" * 60)
print("🌐 PUBLIC TUNNEL ACTIVE — READY TO ACCESS")
print("=" * 60)
print(f"🔑 TUNNEL ENDPOINT PASSWORD:  \033[1;32m{tunnel_password}\033[0m")
print("   (Copy the password above to bypass the localtunnel welcome screen)")
print("-" * 60)

# Run localtunnel foreground/interactive stream
!npx localtunnel --port 3000

## 📖 Clipped Studio Usage Guide & Authentication

### 1. Accessing the Web Studio
1. Click the public `https://*.loca.lt` link generated in **Cell 7**.
2. When the **Localtunnel Reminder** page appears:
   - Paste the **Tunnel Endpoint Password** printed in Cell 7 (e.g. `34.123.45.67`).
   - Click **"Click to Submit"**.
3. The Clipped AI Studio dashboard will load!

---

### 2. Default Login Credentials
- **Email**: `admin@clipped.ai`
- **Password**: `admin`

---

### 3. Studio Creation Suites
- 🎬 **AI Video Generator** (`/create/ai-videos`): Text-to-Video generation using Kling, Luma Dream Machine, and Fal.ai Flux.
- 📖 **Story Series Creator** (`/create/stories`): Multi-part viral narrative generator with hook and cliffhanger optimization.
- 🎭 **Micro-Drama Studio** (`/create/drama`): Episodic character drama series with visual style consistency.
- ✂️ **Shorts Extractor** (`/create/shorts`): Long-form video to viral short clips with retention scoring.
- 📅 **Bulk Content Planner** (`/create/bulk`): 7 to 30-day automated multi-platform social media calendar generator.
- 🚀 **Auto-Pilot Pipeline** (`/create/auto`): Autonomous daily video rendering and publishing workflows.

---

### 4. Troubleshooting & FAQ
- **Connection Lost / Tunnel Closed**: Simply re-run **Cell 7** to start a new tunnel instance.
- **Port 3000 Busy**: Run `!fuser -k 3000/tcp` in a new code cell and re-run Cell 7.
- **Inspect Server Logs**: Run `!tail -n 50 server.log` to inspect live Next.js request/response logs.
- **Cost-Safe Mode**: If API keys are left blank, all engines use in-memory synthetic PCM WAV and royalty-free video clips for zero-cost testing.